# FinDisputeEval CFPB Seed Pool Build v05

This notebook consumes only a persistent EDA v05.1 run and a passed Audit-and-Decision v01 record. It does not read Seed v04. Generation is intentionally blocked when audits, independent annotations, accepted decisions, or machine-readable rules are incomplete.

In [ ]:
# Minimal Colab bootstrap required by the pinned-input preflight below.
from pathlib import Path
import hashlib
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

In [ ]:
# Pin the accepted v05.1 lineage after migration to Google Drive.
# Local VS Code execution keeps the repository paths discovered above.
if IN_COLAB:
    EDA_RUN_DIR_OVERRIDE = (
        "/content/drive/MyDrive/FinDisputeEval/"
        "outputs/data_pipeline/cfpb_seed_source_eda/eda_v051/"
        "run_20260713T145423Z"
    )
    DECISION_RECORD_OVERRIDE = (
        "/content/drive/MyDrive/FinDisputeEval/"
        "dataset/curated/annotations/cfpb_seed_v05_audit/"
        "run_20260713T145423Z/seed_v05_decision_record_v02.json"
    )
    EDA_RUN_DIR = Path(EDA_RUN_DIR_OVERRIDE)
    DECISION_RECORD = Path(DECISION_RECORD_OVERRIDE)
    OUTPUT_DIR = Path(
        "/content/drive/MyDrive/FinDisputeEval/"
        "dataset/curated/seed_pools/cfpb_dispute/seed_v05"
    )

    expected_inputs = {
        EDA_RUN_DIR / "manifest.json": "035dfbdeada1006dfa3fd823f9416b2980a7d54fd6adacc628a58448960c40a7",
        EDA_RUN_DIR / "analysis_ready_corpus.parquet": "3cb41e96cb4a43f1b922b26c226ad35bd85971b1a51b35a38abc1aa42b5602e3",
        DECISION_RECORD: "e1bafa369ada25600da6cff92935417eb5aca4301bceb828bd4ac7658caf9cfe",
        DECISION_RECORD.parent / "fuzzy_duplicate_audit_master.csv": "65ec2c78a62b415f747033455d182cdd8e937e884be2e82cbb48d3be2f65bd8c",
    }

    def migrated_sha256(path, chunk_size=1 << 20):
        digest = hashlib.sha256() if "hashlib" in globals() else __import__("hashlib").sha256()
        with path.open("rb") as stream:
            while chunk := stream.read(chunk_size):
                digest.update(chunk)
        return digest.hexdigest()

    for input_path, expected_hash in expected_inputs.items():
        if not input_path.exists():
            raise FileNotFoundError(f"Missing migrated Seed v05 input: {input_path}")
        actual_hash = migrated_sha256(input_path)
        if actual_hash != expected_hash:
            raise RuntimeError(
                f"Migrated input hash mismatch: {input_path}\n"
                f"expected={expected_hash}\nactual={actual_hash}"
            )

    print("Pinned migrated Seed v05 inputs: PASS")
    print(f"EDA run: {EDA_RUN_DIR}")
    print(f"Decision record: {DECISION_RECORD}")
    print(f"Seed output: {OUTPUT_DIR}")

In [ ]:
from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules
EDA_RUN_DIR_OVERRIDE = ""
DECISION_RECORD_OVERRIDE = ""
RANDOM_SEED = 20260713

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/FinDisputeEval")
    PROJECT_ROOT = Path("/content/FinDisputeEval")
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    DRIVE_ROOT = next(
        (path.resolve() for path in candidates if (path / "WORK_PROGRESS.md").exists()),
        None,
    )
    if DRIVE_ROOT is None:
        raise FileNotFoundError("Run from the FinDisputeEval repository.")
    PROJECT_ROOT = DRIVE_ROOT

eda_root_candidates = [
    DRIVE_ROOT / "outputs/cfpb_seed_source_eda/eda_v051",
    DRIVE_ROOT / "outputs/data_pipeline/cfpb_seed_source_eda/eda_v051",
]
if EDA_RUN_DIR_OVERRIDE:
    EDA_RUN_DIR = Path(EDA_RUN_DIR_OVERRIDE)
else:
    runs = sorted(
        (
            path for root in eda_root_candidates if root.exists()
            for path in root.glob("run_*")
            if (path / "manifest.json").exists()
        ),
        key=lambda path: path.name,
    )
    if not runs:
        raise FileNotFoundError("No persistent v05.1 EDA run found.")
    EDA_RUN_DIR = runs[-1]

if DECISION_RECORD_OVERRIDE:
    DECISION_RECORD = Path(DECISION_RECORD_OVERRIDE)
else:
    audit_root = (
        DRIVE_ROOT
        / "dataset/curated/annotations/cfpb_seed_v05_audit"
        / EDA_RUN_DIR.name
    )
    DECISION_RECORD = audit_root / "seed_v05_decision_record_v02.json"

OUTPUT_DIR = (
    DRIVE_ROOT / "dataset/curated/seed_pools/cfpb_dispute/seed_v05"
)
print(f"EDA run: {EDA_RUN_DIR}")
print(f"Decision record: {DECISION_RECORD}")
print(f"Seed output: {OUTPUT_DIR}")

In [ ]:
import base64
import gzip
import hashlib

SAMPLER_SHA256 = "1f0fcfac9dda5eb8b122fc3007b43ad3f185c3667b9c309cca89cfeef7c0d23a"
SAMPLER_GZIP_BASE64 = "H4sIAAAAAAAEAK1cbXPbyJH+zl+BxdWmwJhCKGe9l7DMOI4t1+lqYzu2d+suFAoFkUMSaxCA8SKJK2t/e7p73oEBSW2tyiWTg5me7p6e7qd7BvJ9/zVbpnVa5GebpGEr79Wb9//wPjL4dDN95tXJrsxYFY5Gn7ZMfvPSvGF5A2OSLNt726T28sLbsWZbrIqs2KTLJPNWbJ20WVOH3mXjLYu8bnes9oocBiSjMqlrMUPSrtIGenMmvIoti2rl3W6LmnnJcslKZKoqbmsk0iRp7u2S5TbN2VnFklVynbHR/35899ar2ozxyXJ2AzziU5gPmd2yKm1qrwEJbqbfeTUKV7GMJTULR77vj0brqth5cbxum7Ziceylu7KoGi/J86JJUNB6NBJtIO02S6/l15/rIpefK8YJrZImWWYoYi0pqSbVgzXpjhmP6fvEw9+/FLmgVCYNTia7vYev/EGzL9N8I9tf5nvFX97uSlAxLEkpm8okX0ED/CtXo9How8W/frz8cPE6fn3x6vLj5bu38csPFy8/enPvfuTBj8/ukmUTr9oyg5VsWO1PeHvDYPmhIV4nuzRL9YN1+8sve8eAim3SumGV/A7rso/rLfAUZ0W+0d1WMCFoWTaUaSo/Zkm+aZMNU99B7BaIpst4zRJcLjUbWSc8jr+0sGrY/IDC/gCyXcSfLv7vU/zq3Q8//vOtIWmeVBWs7w2L2/y2SsqSrSQ1/QgMkjlak7zI0dL7jzJ2Jx48jD6++/HDq6HpX/FtUYFpo2ZhW3mKSp9sldz2G39PvndJs9weEoce/PvyPfC7YiTe+8vL+P3LT58uPphyXfzz5eUP/gw2RIiSpRkLKv/qevHy7N/Ts7+G8bdPzqInf5df4fNViF+i+6eTh6trf4IDL8divvf/8+7thU2MHuBP5Qcvnn9ztRoHL2ZXT16cL87Cqzp6MX6B34MXV6v7Pz9cjV/IZvouvsDn7x6CFzjYJ3pyvo8f33ZZF5PQ8DP4/ZR+q+Fy5MtXr979+PbTIK9X18AW+LSizZuv8H/zdZlUQLb+I7TDvr1m1de8uApffP0vbIQpvuf60DRALxOL28v3P33X1zSKv7o/n4D04Rh1IL5cI6+wK0Z/1w6JfpPH/2n6DCxynW5mRBt2ZVy1ebxKqxl3PdgqPXXMPXWMLsp4XLRN2TadMRU4oGIXo+OdYfAAS3k6ffr99L/P/zwSRNfgfsuibuI0T5s4DmqWrcfe2d+8t+AMZ0oB2BwajAElnCXoto9DcAxFdsOCsT3Wxb5FxNVhkJqW1qKhm82RoxGKWW+Tp8++j9e4VFp3E2+5bfPPcZ3+wqSKzr3nz0FPpIW6qbgSVumG1fhUxKGQ0xOM3aYgDfHB2S5Klgd+BcuO3h+IsGSnlXm7BSb4xN5sLh6HGDgDzc1Y99fzh22JIYt341NXDJxxLp9v2R3/pOSG1c8Yefn4M9sHyhYm2vnFKTQBF7bEZbLPimQFIq/9exz28PXeHPLghyxHhxT4bbM++4tv8dNRkyA2djCI7bFEHDHBCWOBiKcGIhxbrNJlswD2Jhh3o4lnf49mYnrCMXMCCCHSrgNjYVDJccPumoB4h6g1l9xz9tM1YKpGkAk3rIFl5JAlRpwGPvL+YczbOZiCljdJVpvrVSUpgKifkqxlF1VVVIGvcJ2g5SEtAeAQJxAhxYA5eV201ZLFuMUgiuMGAZv6Zg4xffosPPdPnPRLm8J+8BKPRnVRn5hZttagPZMF1Q6iLiLe93ofJ6BLDD2AEReqD7X60QyRo7cGFIj/A3ZUNB5o+C6ta8RRYPwAStgqGMJGZ7Dfm0DMppdIjD8g/Vrha2kSaa2mRWL1zLsX3x+k6aLtzTp2hSJyplEcEhrkOcy2aQsg/1yqa4G/IvUMBIFpAuhiKzquAfy2qG4fzBKM+JZVAV91uU982zscEj/lRiZHgtjIhZQZf25wnDe3mRFonzZkjJtJsgO90tJwyGLHEJHTuZLJS3cWB3tNte/QxXXiurQ3OvGgB7I7FJl3wFwFJgd3RaygW2b44SjHlzlQTVd9TskeJLMeZQdE0fSC3PQmnGPpkIkewnkihdlVBTA8cFpfJxSLnbxCV14r3Ic//eQBNAmLeBeXrIr5Q/TJ/oNGNY7EwhxEjXtrQC/hwP4Ag7M9yrFOqx3DyJ1WtTVM5SPUPUODBk8vGsnOC0gSq1hkI+bQbuqCFIw2mb+IjtjFRUVnOkrAomwzSjHjsiowXRN0kusCsHezBX+5LbKVixqmSTND91JGdhdXaf05tpIqowO5fh5+QF1Zu+v14aoUpDrZmeq0An4h/3DRq42+JscqmzMXgHsZdlj/rrwPiaikT7Wao7opYU9bZZXuElxFhowA3OkKyfIqXW53DKCGSiutDoia6hqEB+zqIgDPYXnX+/h631eK7dAnHm2nNLf3V5jC/qgDw5n3whaN4zHKcEpjyzn2YhX+OBwNdyXkKsxYhVP0QxVfKXIBYFhYnsFVBIeXtZghKvVg3ccfkvb+0DZbnLjNooFtthjaSVFvJy0cmydyW+/CttTI7WTcvR76C4o6AW2gMkjP1hohEtOLuoAOEUU66CzUb/cfWNcf87otS7IXMQ4wU4fuN9WDEU7Ce2iTKw0qokCBoIxGkdIiiYTk44XbvyiOXfWYQ+AR6IYuigpOgBJK8Jw3IBU+9m5SdutbGBoCfQ4+Jl9CwqW4dHs4QPPXRZEdxNDIkXO0t2shM7tmRIIlueZCT3vQZ0ZSqE+VCWEOIGqaEWl6mqbHofoZaUOSVqzoeoS5Yj2rJ5Ant7EaAmk8TxSOqZGXKA6IYPvID+Q5gHUwPe4wzqhQTHQ9ZKyWLlG2qbkUJZn2iQd6w2PmCLhMukbDO0SLAzE4soxoGk6953MneWg+D6eH1kuNCgenM4ynzVegisV04p1HtiUrAYSHAf77KCY6zAnvFvbGcQZwFmAC8Fizd86tfKAxtwrgB6eWI8PusGMzG/tXMuEAA8BPHwvAfs6g38H97CAW9iip5UmI4DEGe/g0WgzA01M8Tpda6CQ14H6scNuJtg6s7oLqR5B6F6gPRjlQV5pbCIWCzth77p2fCEt4SDJWoyzqFKvUdCa1gT3BpeZYTceqLhCMlG6EVgIXFHSgu744nKKSZHpUkg4vHZHyIj/L2YafAQyZmZjTgU5FPeqgPXUZ6FHR+s294vpntjQMPsn3AYrNc1wUmNTIc3dQ5AHWQuoEJnGQuws10Otw8yjVmLhbugGML9jd2ecxGjMGGraI/JH/kj5CVT4lcuVxb10lOzbzylX4OmmSN/htAnl9/ZnaPrIK9tVEYDQqiOKBSFKLL6NOLp6s17BADMMc0vD+wOmDdGjIMtKF7EsgEfpYcxFmxXIhKaC9m2OAIv94Uv+Y80jD+EchP/dWLEs36TX4jGYviqvDunCWILjcZkdZi6jbDCvinMFlUe6DsfFk0RNK6mGokymJz3tlLN802w4A7iYr3Kn0llvPMul8X/hf2oQ0sgRoxYsxmigs2jrNsjwJeGFXjza4caVHkbNuIfzy+Hdjkst8Oo9D2ZomZnKokZat9C7KfJQkmihEsSVD+AZxywR94aaxwamAjhY3J8BGEsRkwB5/Qk6qh3Q6//aFJNyuwP7w2pkJ3rGkuEPT5E0CPYn8MB7rEsURKPkI8X6V8mWppACyYUwIeiz0pTTZGEzqZSdBPoYoEgvaTpE1rLZEPoLcf4vMcrjenV3Z1QR92TU/vNhuyd+p3owdNY5BPai6MzIp4sAyKTXWjDdV0ZaPDgKT3lnyiWEhqet0Y+jTPo6ci5D5St3DuHwNOtklpe0MsmR3vUrEGUP3SNPgbEJnGBwiGf6AKyjEol0ssJBvExGxmYNvUBjwryFzH6/LBNbG68KOOBzvU+ljeIOMwPA6+dWsAEpHAGWQtXB7HzW9lozi0msY14XstT+2V4x/IKjRsf+QrOZ6H/jqGgsvofDjXX8cLtsd3a8IxmHWBIp5PkP0+HmEtGg8lAn2ptDqsOdw27/O2XhSt8xa2prD+2Akt2tvK9CT4esYk6GdITAzpomBWcXppaYmTOfyEIO8HAhdYuvADihxJ5Ksfm5XtO6CEjHlOrYTnGhaJ1eMuimxcbbsnN273bLcUxJ6PGOGlJLQaQreq7NLiSfzcohmss879gOfC4G6LrK2YWbSaxJyLVZYAhfgcP6ke7rpsztwuVY+zXX0Js3Y26J5g3Ujrio1RouCccgo/a34BQSY+ksLS9YZIKbXU9ftep3eqQPgOcDiUIzVemOAIRTlZX2jqUJK2uxLNsdwkiLMg6SbQSzgN0TjPJlzAGLFD34AQgcL5m2POEFDslrwwhj4rxxQagyJGiAl4eD8gUN+RZyfmJB+QlEpfdzZ/kttbB6Z5BkS8/Byj3l2Imj3j0+0Qc75MtnOiJoWbtkiCGmo1QAvzOCuksvDg7+1XcDnVy3VM/b8mO3cjzr2HglTQVs0PQ38ElcPhLtZg3T5KhAhsHdbR1MJQbdiiXnvidc5Fycbw64LekBFZ8fxvd1nzue3GsddP2U91Zy3OWKqjK0bmVanm22jpbDvmmG/uCqKRvSjz3J+fDie8M/01BJLDUWR9GCnXBB5A+dUYxR2l+YDT0eqhIXPxSOswCij0oa0OL6HuAFEWLWr6J5THYBs7I7vzYmXg/XNUUGGA+L6RO8v1EGBgLTRCemDuflAQBRVBN7nECqzERkthwG95NUWUaxKpWVwZ7X2cS9mrCny2T11efANH2QDJ3kbhcMkzbZkeOKda99xAjgapNTDSb4DUh8BRwNK7WIXA7SIUhVPOkVCB6ldgOhqJoqA9k5Xa4sKxm663owvBzT7CV5fh2lz1Lt5UbhfGVa0xAiIONfgXhcfLl6/fPXp4nV8z0k+RHQ5GDsP5xllxXCVY3lfjmzvCMCSJ1pukJXmdBAWc2UgcAJW8ZBuEGMNG764SCdOMedHD06lSVk8mL7GJiiOWYVFyMBj+R1HHPsgrgMaZ4ZG/IK4ZU1iXoqyymc43Ny51ihVbQAYaMYuKxwcO1jsSHJocrOVfEbPuF37Cg9TtXcR+psvhE7QuKV6cs91tx83v+rAaRp1BLqYVc99yCb05fyxsltZRMJgzyvMKdZE6K2bIwac88u03MRwbDKjKjSasmCgmzkfTBDwVHNK6R4ZMJW13elAuMULw9OxaeRE4W9zLAIG1GkglTDH9GAaYjMuylggNTnmNyC1j6KCL5RD0F/OSHfmAJCXrQOi3RbVZ85Wj2PxaOHz1Wp3ZH6ylU+kYJoEv2NrG2w2gf/VD38u0nziJYDw5+cSGYKXrg1yxiT8HCXmXQK8TIzBGRKvmyTN8I0ooNwUwu+Z5BZ+umJJRmzKFj0oghwEF0zMOPb+6OXWYCz0ENYlAnkZrrMClNuhrTY3mNj4wHjRajbjtTfNDuxarGIZSqnYLkFQgrftc7AJjM0O4hA9doEwD0izqHuXSxjtGGnXZpJ6yXK6m83zk5AAkYpyYj1wo9MsllfWrOJGst3WNaRIn83OnJMwaRaCJirCUNZzdw+tKpv+MXJP5t55x49KZs/kI4igZH4LfVJarNewCSdeoKhWxS2BKpbjW0z0QgCfGkEkvrNnHfU1yWcmARXe1Tb1bmoD+9kCEbDRe4HAj2tfsC+St3FkESBp8Ngc1jMgaqHwqvkcp9NlxQYYmhue0nsi5B5b0QKcJoAs4D4g0hOPe/SYQ2aEBmMeSmQE0WxKd3/dptmKnzbfTJ8FS/7mjf0iDjlojUiwECHWulNKEOVS0JHrNQZO2/1qC1E7eBdYlCRXSQyJaLrGo3BRzxB0zfdx/kSpOPUK6bZ2b7B9VbpH9qRXI8xRHJj/fi8lAOT0JG1Vji3pwDNWGhRpRK+kY70soeSS1UldUx2gB7yb7wf1lHPw4Lz7osGqYPw6Gb3XR6+/8lozSO4QkgdARKSd6pB7lRNAKfs6rfGwdIXFQxytqkJKVAwo/BmtzNPps/jZX/56MFRfCO1g5wl0lpyhQ5l4GxDo3qA6mzxYAiz8HVjvlkIMSAJxUL5ca8I63pNbPFsyyLJIXnz9JZoomLYsGKwjhKW2WdKm5ngtXDVYcFkTUf/b/z/7dtfV4dCZN+8gdqvyKPzy21wMJ/cmWRw+xI/sdFoTcR2zyOdi5oncuoancxI6XK82B/TOlvp3oCbGAgw6JPMU6bRbPALRzQnvuu9/iEKoOrClLIHzTdqWXxa+cajLV4+QCB/OLwihXZ2A1O0JtejGlSHHlaPIOGXjcvWUZh4zGboy7tscjN3qZps4ZQP3Bz4Alt8O5JoLDsYxCB26djSWSb2o4uodrs4l667WrRD965E1sDr/wdO9bYkECrDaNLrQVLrqkuDAhg0nrnNf1EnvGcEepez+c3eKKWzbyir1LDIVivrk+sYCSOZ8Gk+nUwVp7FHdc0qtoe7ZgUA+XRW6QJCJ67r99VpgMU5ZhiOdFJUTxuvkioGF2D8Tg3Q0yAUOF/405RkIbzm9uugv1+U1grXYBx123izVZcbu66jWy6aL2fn3kSkUP2G3NslJYcA6mo8MWo/wT73J3W6qdwnS9FKn221vNof5Os32KZntxGEKzkofP4A377BMrOqZaRNaY05K9PhEWtKZy1druX3RKnJDc4cW086NHeKg8uvJZPi9/WENdYDQEUuTrwGMo16M5xoxjMbQDD83MBZNrK1+OT7cfYbf4iinJlXCVsZjzrj4bGoWYEHnLUPiUoLNWZ80IlTarTK/Ush00iGCuUh2Ggne1bz7ZNwAU7zYYeE4WeMeWf9Q1bx0Y/vP3zqbpnJ4NrFbH6Pj2HAULn1Lu3wcUTmqQ/FBOQLE+PokG0xlYZsHbVl1kjW2xuGSWoP4GsOQokpBR3OfQ1M8NQVAwaSVgreDBC+pl2lqku34AAdnDpNx8df3BA5aDoNwykqr4tKSvcROPqTyHbN3V9M13kj6jd0rRMJ3B6w/N2UayxLySkyRIfWCfjKHC/PiNpB/IymEZ3jcXcBqQI4bWDfUdATRZtaDznwXk/+JrWRbZOyzI+l4n0o3pXESOlSPMbcgLyNg3gsEjJzXnFZanO4kmgZch9FRtzo2vtGPt1jaRfePf3aGkkndzvcxvWdrveEq9jf9Ga9DfpxmYxY+tr20q4PTCxuKkDvoqC+1deIa1nOKbvU4HB3RFs1jUzXSoBZKeZGlopwOfLqvLeOFKrq3EHSMhi8PXztD6wsk00kTfKdtuugh6IM8mr8hxG8ENbjfQgEIjfea1SfEgzmlmFQspKN38BsyTXRooFNVtGOCCgeO6qJdP7ytYApeQFTMUMFx1e7KOpCdMWmo8a+9GU6cOzDw+k/H+NiuPhpIRnhAVUCjixomE1apGDuP/gP9Vn4IaU8AAA=="
sampler_bytes = gzip.decompress(base64.b64decode(SAMPLER_GZIP_BASE64))
assert hashlib.sha256(sampler_bytes).hexdigest() == SAMPLER_SHA256
package_dir = PROJECT_ROOT / "src/findisputeeval/curation"
package_dir.mkdir(parents=True, exist_ok=True)
(package_dir.parent / "__init__.py").write_text(
    '"""FinDisputeEval reusable pipeline package."""\n',
    encoding="utf-8",
)
(package_dir / "__init__.py").write_text(
    '"""Seed curation workflows."""\n',
    encoding="utf-8",
)
(package_dir / "cfpb_seed_v05.py").write_bytes(sampler_bytes)
print(f"Bootstrapped gated Seed v05 sampler: {package_dir}")

In [ ]:
import subprocess

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pandas>=2.2,<3",
        "numpy>=1.26,<3",
        "pyarrow>=16,<22",
    ]
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
# Re-apply the pinned paths after the shared local/Colab discovery cell.
if IN_COLAB:
    EDA_RUN_DIR = Path(
        "/content/drive/MyDrive/FinDisputeEval/"
        "outputs/data_pipeline/cfpb_seed_source_eda/eda_v051/"
        "run_20260713T145423Z"
    )
    DECISION_RECORD = Path(
        "/content/drive/MyDrive/FinDisputeEval/"
        "dataset/curated/annotations/cfpb_seed_v05_audit/"
        "run_20260713T145423Z/seed_v05_decision_record_v02.json"
    )
    OUTPUT_DIR = Path(
        "/content/drive/MyDrive/FinDisputeEval/"
        "dataset/curated/seed_pools/cfpb_dispute/seed_v05"
    )
print(f"Build EDA run: {EDA_RUN_DIR}")
print(f"Build decision record: {DECISION_RECORD}")
print(f"Build output: {OUTPUT_DIR}")

## Release-gated build

The accepted v05.1 decision record is pinned above. The build remains fail-closed if its release gate or machine-readable rule contract is invalid.

In [ ]:
import json
from findisputeeval.curation.cfpb_seed_v05 import (
    SeedV05Config,
    build_seed_v05,
    sha256_file,
)

paths = build_seed_v05(
    SeedV05Config(
        eda_run_dir=EDA_RUN_DIR,
        decision_record_path=DECISION_RECORD,
        output_dir=OUTPUT_DIR,
        random_seed=RANDOM_SEED,
    )
)
manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"))
for metadata in manifest["outputs"].values():
    path = Path(metadata["path"])
    assert path.exists()
    assert path.stat().st_size == metadata["size_bytes"]
    assert sha256_file(path) == metadata["sha256"]
print(json.dumps(manifest, ensure_ascii=False, indent=2))

## Completion boundary

Seed v05 exists only when this notebook writes a hash-verified `seed_v05_manifest.json`. A pending decision record, partially completed audit, or manually edited output without a matching manifest is not a Seed v05 release.